## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

### PART A: Divide our documents into chunks

In [3]:
!ls 

anscombe.json		      llm_engineering	     README.md
california_housing_test.csv   mnist_test.csv
california_housing_train.csv  mnist_train_small.csv


In [4]:
# %cd sample_data/
# !git clone https://github.com/hdmeyer/llm_engineering.git

%cd llm_engineering/
%run week1/set_ollama.ipynb

/content/sample_data/llm_engineering
[Errno 2] No such file or directory: 'llm_engineering/'
/content/sample_data/llm_engineering
Using CPython 3.12.12
Creating virtual environment at: .venv
Resolved 267 packages in 5.74s                                       
Prepared 260 packages in 1m 49s                                          
Installed 260 packages in 2.31s                             
 + aiofiles==24.1.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.2
 + aiosignal==1.4.0
 + annotated-doc==0.0.3
 + annotated-types==0.7.0
 + ansi2html==1.9.2
 + anthropic==0.72.0
 + anyio==4.11.0
 + asttokens==3.0.0
 + attrs==25.4.0
 + backoff==2.2.1
 + bcrypt==5.0.0
 + beautifulsoup4==4.14.2
 + blinker==1.9.0
 + brotli==1.2.0
 + build==1.3.0
 + cachetools==6.2.1
 + cbor2==5.7.1
 + certifi==2025.10.5
 + charset-normalizer==3.4.4
 + chromadb==1.3.4
 + click==8.3.0
 + coloredlogs==15.0.1
 + comm==0.2.3
 + contourpy==1.3.3
 + cycler==0.12.1
 + dash==3.2.0
 + dataclasses-json==0.6.7
 + datasets==3.6.0
 

In [5]:
import subprocess

# Start the command in a background process
process = subprocess.Popen(["ollama", "serve"])

# The kernel can continue execution while the process runs in the background
print("The 'ollama serve' process is running in the background.")

The 'ollama serve' process is running in the background.


In [6]:
!pip install ollama
!ollama pull llama3.2:1b

In [7]:
!ollama list

NAME           ID              SIZE      MODIFIED       
llama3.2:1b    baf6a787fdff    1.3 GB    38 seconds ago    


In [14]:
!pip install langchain_chroma langchain_openai langchain_chroma langchain_huggingface langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.3
    Uninstalling langchain-core-1.6.3:
      Successfully uninstalled langchain-core-1.6.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 w

In [11]:
!pip install -U \
  opentelemetry-api \
  opentelemetry-sdk \
  opentelemetry-exporter-otlp


In [15]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

/tmp/ipykernel_1028/3216454956.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [23]:
!pip list | grep -E "chroma|opentelemetry"

chromadb                                 1.5.9
langchain-chroma                         1.1.0
opentelemetry-api                        1.45.0
opentelemetry-exporter-http-transport    0.66b0
opentelemetry-exporter-otlp              1.45.0
opentelemetry-exporter-otlp-common       0.66b0
opentelemetry-exporter-otlp-proto-common 1.45.0
opentelemetry-exporter-otlp-proto-grpc   1.45.0
opentelemetry-exporter-otlp-proto-http   1.45.0
opentelemetry-proto                      1.45.0
opentelemetry-resourcedetector-gcp       1.14.0
opentelemetry-sdk                        1.45.0
opentelemetry-semantic-conventions       0.66b0


In [ ]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


: 

In [ ]:

from openai import OpenAI
db_name = "vector_db"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
MODEL = 'llama3.2:1b'

In [34]:
db_name = "vector_db"

In [20]:
%cd week5/

/content/sample_data/llm_engineering/week5


In [21]:
# How many characters in all the documents?

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


In [26]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for llama3.2:1b: 63,721


In [27]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [28]:
documents[1]

Document(metadata={'source': 'knowledge-base/employees/David Kim.md', 'doc_type': 'employees'}, page_content='# HR Record\n\n# David Kim\n\n## Summary\n- **Date of Birth:** September 22, 1992\n- **Job Title:** DevOps Engineer\n- **Location:** New York, New York\n- **Current Salary:** $118,000\n\n## Insurellm Career Progression\n- **August 2021 - Present:** DevOps Engineer\n  - Manages AWS infrastructure for all Insurellm products\n  - Implemented CI/CD pipelines reducing deployment time by 60%\n  - Led migration to Kubernetes, improving system scalability and reliability\n\n- **March 2019 - July 2021:** Junior DevOps Engineer\n  - Assisted with infrastructure monitoring and incident response\n  - Automated manual deployment processes using Jenkins and Terraform\n  - Supported senior engineers in cloud infrastructure management\n\n- **June 2017 - February 2019:** Systems Administrator at CloudFirst Solutions\n  - Maintained Linux servers and network infrastructure\n  - Provided technica

In [29]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 413 chunks
First chunk:

page_content='# HR Record

# Lisa Anderson

## Summary
- **Date of Birth:** November 18, 1985
- **Job Title:** Marketing Manager
- **Location:** Austin, Texas
- **Current Salary:** $105,000

## Insurellm Career Progression
- **April 2019 - Present:** Marketing Manager
  - Leads demand generation and content marketing teams
  - Manages $2M annual marketing budget
  - Oversees digital campaigns, events, and brand initiatives
  - Increased qualified leads by 65% since 2021

- **January 2016 - March 2019:** Senior Marketing Specialist at GrowthTech
  - Managed B2B marketing campaigns for SaaS products
  - Specialized in content marketing and marketing automation

- **June 2012 - December 2015:** Marketing Coordinator at Enterprise Solutions
  - Supported marketing team with campaign execution
  - Managed social media and email marketing programs' metadata={'source': 'knowledge-base/employees/Lisa Anderson.md', 'doc_type': 'employees'}


In [32]:
chunks[100]

Document(metadata={'source': 'knowledge-base/employees/Michelle Rivera.md', 'doc_type': 'employees'}, page_content='- **2019:** Rating: 4.0/5\n  *Solid first year at Insurellm. Quick to learn complex insurance domain and establish design process.*\n\n## Compensation History\n- **2023:** Base Salary: $118,000 + Bonus: $16,000\n- **2022:** Base Salary: $112,000 + Bonus: $14,000\n- **2021:** Base Salary: $105,000 + Bonus: $12,000\n- **2020:** Base Salary: $98,000 + Bonus: $10,000\n- **2019:** Base Salary: $92,000 + Bonus: $8,000')

### PART B: Make vectors and store in Chroma

In Week 3, you set up a Hugging Face account and got an HF_TOKEN

At this point, you might want to add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).

In [35]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectorstore created with 413 documents


In [ ]:
# Let's investigate the vectors

collection = vectorstore._collection # collection is the table
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 413 vectors with 384 dimensions in the vector store


### Part C: Visualize!

In [37]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [52]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

# fig.show()
fig.write_html("chart1.html")

In [53]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

# fig.show()
fig.write_html("chart2.html")

In [59]:
from google.colab import files

files.download("chart2.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [61]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [62]:
!cp chart2.html /content/drive/MyDrive/

In [63]:
!cp chart1.html /content/drive/MyDrive/

In [65]:
from google.colab import drive

drive.flush_and_unmount()